# Nook 3DGS Pipeline — Free GPU Test

Runs **ffmpeg compress → COLMAP → splatfacto → PLY export** on Colab's free **T4 GPU**.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your property video to **Google Drive** (any folder)
3. Run cells top-to-bottom — the second cell will ask you to authorize Drive access

## 0. Confirm T4 GPU

In [ ]:
import subprocess, shutil

if not shutil.which('nvidia-smi'):
    raise RuntimeError(
        "No GPU detected.\n"
        "Fix: Runtime → Change runtime type → T4 GPU → Save, then reconnect."
    )
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv'],
    capture_output=True, text=True
)
print(result.stdout)
if 'T4' not in result.stdout:
    print("WARNING: not a T4. Proceed anyway but you may hit memory issues.")

## 1. Mount Google Drive + set video path

After mounting, update `GDRIVE_VIDEO_PATH` to point to your uploaded video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
# After mounting, browse Files panel (left sidebar) to find your video.
# Right-click it → Copy path, paste here.
GDRIVE_VIDEO_PATH = "/content/drive/MyDrive/YOUR_VIDEO.MOV"
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(GDRIVE_VIDEO_PATH):
    raise FileNotFoundError(f"Video not found at: {GDRIVE_VIDEO_PATH}\nCheck the path in the Files panel.")

size_mb = os.path.getsize(GDRIVE_VIDEO_PATH) / 1e6
print(f"Found video: {GDRIVE_VIDEO_PATH}  ({size_mb:.0f} MB)")

## 2. Install COLMAP + nerfstudio + pre-compile gsplat

~20–25 min total. The gsplat CUDA compile (26 .cu files, MAX_JOBS=1 to avoid OOM) happens here so training starts instantly later. Do this step, then you can safely leave and come back — the compile cache survives as long as the runtime stays connected.

In [ ]:
import subprocess, os

print("Installing COLMAP + ffmpeg...")
subprocess.run(['apt-get', 'install', '-y', '-qq', 'colmap', 'ffmpeg'],
               capture_output=True, check=True)

print("Installing nerfstudio + gsplat...")
result = subprocess.run(['pip', 'install', '-q', 'ninja', 'nerfstudio', 'gsplat'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr[-500:])
    raise RuntimeError('pip install failed')

import torch
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}")

# Pre-compile gsplat CUDA kernels now so training starts instantly.
# MAX_JOBS=1 = one nvcc process at a time; parallel compile OOMs on Colab 12GB RAM.
# Takes ~15-20 min but is cached for the rest of this session.
print("\nPre-compiling gsplat CUDA kernels (15-20 min, one-time per session)...")
env = os.environ.copy()
env['MAX_JOBS'] = '1'
result = subprocess.run(
    ['python', '-c', 'import gsplat; from gsplat.cuda._wrapper import _make_lazy_cuda_obj; print("gsplat ready")'],
    env=env, text=True, capture_output=True
)
# Force the actual compilation by calling a real gsplat op
result2 = subprocess.run(
    ['python', '-c',
     'import os; os.environ["MAX_JOBS"]="1"; import torch; '
     'from gsplat import project_gaussians; '
     'means = torch.zeros(1,3,device="cuda"); '
     'print("gsplat compiled OK")'],
    env=env, text=True, capture_output=True
)
print(result2.stdout or result2.stderr[-300:])
print("All done — training will start instantly in step 5.")

## 3. Copy from Drive, then compress (GPU-accelerated)

Three things in one cell: copy the video to local disk (Drive mount is slow), download a GPU-capable ffmpeg, then compress using the T4 to decode 4K. Same `libx264 -crf 23` encoder as the app, so the output matches — just much faster. If it's *still* slow, flip `USE_NVENC = True` for full-GPU encode (~30s, near-identical quality).

In [ ]:
import subprocess, shutil, os, time

os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/work/data/images', exist_ok=True)

# --- 1. Copy from Drive to local SSD ---
local_raw = '/content/work/input_raw' + os.path.splitext(GDRIVE_VIDEO_PATH)[1]
if not os.path.exists(local_raw):
    print("Copying from Drive to local disk...")
    t0 = time.time()
    shutil.copy2(GDRIVE_VIDEO_PATH, local_raw)
    print(f"  Copied {os.path.getsize(local_raw)/1e6:.0f} MB in {time.time()-t0:.0f}s")
else:
    print(f"Local copy exists ({os.path.getsize(local_raw)/1e6:.0f} MB)")

# --- 2. Get GPU-capable ffmpeg ---
ff = subprocess.run("ls -d /tmp/ffmpeg-*/bin/ffmpeg 2>/dev/null", shell=True,
                    capture_output=True, text=True).stdout.strip()
if not ff:
    print("Downloading GPU-enabled ffmpeg (~15s)...")
    subprocess.run('wget -q https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/'
                   'ffmpeg-master-latest-linux64-gpl.tar.xz -O /tmp/ff.tar.xz', shell=True, check=True)
    subprocess.run('tar xf /tmp/ff.tar.xz -C /tmp', shell=True, check=True)
    ff = subprocess.run("ls -d /tmp/ffmpeg-*/bin/ffmpeg", shell=True,
                        capture_output=True, text=True).stdout.strip()
ffprobe = os.path.join(os.path.dirname(ff), 'ffprobe')
print("ffmpeg:", ff)

# --- 3. Probe source ---
probe = subprocess.run([ffprobe, '-v', 'error', '-select_streams', 'v:0',
    '-show_entries', 'stream=width,height,nb_frames', '-show_entries', 'format=duration',
    '-of', 'default=noprint_wrappers=1', local_raw], capture_output=True, text=True)
print("Source:", probe.stdout.replace('\n', '  '))

# --- 4. Extract 100 frames directly as JPEGs (skip re-encoding entirely) ---
# 100 frames is enough for a quality test and keeps COLMAP under 5 min.
# We skip video re-encoding and go straight to frames nerfstudio needs.
print("\nExtracting 100 frames...")
t0 = time.time()
total_dur = next((l.split('=')[1] for l in probe.stdout.split('\n') if l.startswith('duration')), None)
fps_target = 100 / float(total_dur) if total_dur else 0.5
r = subprocess.run([
    ff, '-y', '-i', local_raw,
    '-vf', f'fps={fps_target:.4f},scale=960:540:force_original_aspect_ratio=decrease,scale=trunc(iw/2)*2:trunc(ih/2)*2',
    '-q:v', '2',
    '/content/work/data/images/frame_%04d.jpg'
], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError('frame extraction failed')
import glob
n = len(glob.glob('/content/work/data/images/*.jpg'))
print(f"Done in {time.time()-t0:.0f}s — {n} frames at 960x540")

## 4. COLMAP structure-from-motion

Runs COLMAP manually with CPU SIFT (nerfstudio's auto-invocation crashes without a display). Uses sequential matching (video-appropriate, O(n) not O(n²)). 100 frames keeps this under 5 min total.

In [ ]:
import subprocess, os

IMG    = '/content/work/data/images'
DB     = '/content/work/data/colmap/database.db'
SPARSE = '/content/work/data/colmap/sparse'
DATA   = '/content/work/data'

if os.path.exists(DB): os.remove(DB)
os.makedirs(SPARSE, exist_ok=True)

print("1/4 Feature extraction (CPU SIFT)...")
subprocess.run(['colmap', 'feature_extractor',
    '--database_path', DB, '--image_path', IMG,
    '--ImageReader.single_camera', '1',
    '--ImageReader.camera_model', 'OPENCV',
    '--SiftExtraction.use_gpu', '0'], check=True)

print("\n2/4 Sequential matching (~2 min)...")
subprocess.run(['colmap', 'sequential_matcher',
    '--database_path', DB,
    '--SiftMatching.use_gpu', '0',
    '--SequentialMatching.overlap', '10'], check=True)

print("\n3/4 Sparse reconstruction...")
subprocess.run(['colmap', 'mapper',
    '--database_path', DB, '--image_path', IMG,
    '--output_path', SPARSE], check=True)

models = sorted(os.listdir(SPARSE))
if not models:
    raise RuntimeError("No sparse model produced — check mapper output above")
print(f"Sparse models: {models}")

print("\n4/4 Converting to nerfstudio format...")
result = subprocess.run(['ns-process-data', 'images',
    '--data', IMG, '--output-dir', DATA,
    '--skip-colmap', '--colmap-model-path', f'{SPARSE}/{models[0]}',
    '--verbose'], text=True)
if result.returncode != 0:
    raise RuntimeError('ns-process-data failed')
print("Done.")

## 5. Train splatfacto

~8–15 min on T4. `--downscale-factor 2` uses 960×540 images (4× less VRAM than 1080p — avoids OOM on Colab free tier). Use 3000 iterations for a quick quality check; bump to 7000 for a production run.

In [ ]:
import subprocess, os

# MAX_JOBS=1: gsplat compiles 26 .cu files before training starts.
# Default is parallel (one job per CPU core) which OOMs on Colab's 12GB RAM
# since each nvcc process eats ~2GB. Single-threaded compile takes ~15-20 min
# but only happens once per session — result is cached after that.
env = os.environ.copy()
env['MAX_JOBS'] = '1'

SPLAT = '/content/work/splat'
DATA  = '/content/work/data'

result = subprocess.run([
    'ns-train', 'splatfacto',
    '--data', DATA,
    '--output-dir', SPLAT,
    '--max-num-iterations', '3000',
    '--viewer.quit-on-train-completion', 'True',
    'nerfstudio-data',
    '--downscale-factor', '2',
], text=True, env=env)

if result.returncode != 0:
    raise RuntimeError(f'ns-train failed (exit {result.returncode})')
print('Training complete.')

## 6. Export PLY

In [ ]:
import glob

configs = sorted(glob.glob(f'{SPLAT}/**/config.yml', recursive=True))
if not configs:
    raise RuntimeError('No config.yml found — did training complete?')
config = configs[-1]
print('Using config:', config)

OUT = '/content/work/output'
os.makedirs(OUT, exist_ok=True)

cmd = ['ns-export', 'gaussian-splat', '--load-config', config, '--output-dir', OUT]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f'ns-export failed (exit {result.returncode})')

plys = sorted(glob.glob(f'{OUT}/**/*.ply', recursive=True))
if not plys:
    raise RuntimeError('Export succeeded but no .ply file found')
ply = plys[-1]
print(f"PLY: {ply}  ({os.path.getsize(ply)/1e6:.1f} MB)")

## 7. Download PLY + view in SuperSplat

Downloads to your machine. Drop the file into **https://superspl.at/editor** to inspect quality.

In [ ]:
from google.colab import files
files.download(ply)
print('File downloading. Drop it into https://superspl.at/editor to view.')

## (Optional) Push to Vercel Blob → wire into the app

Set `BLOB_TOKEN` to upload straight to the production store. Then use the Supabase dashboard to set your test tour's `ply_url` and `status = complete` to test the viewer end-to-end without a Modal run.

In [ ]:
BLOB_TOKEN = ""  # paste BLOB_READ_WRITE_TOKEN here to enable
TOUR_ID = "ff6e9a13-99c9-4d99-9e33-db5a62afe8af"  # test tour

if BLOB_TOKEN:
    import requests
    remote = f'tours/{TOUR_ID}/splat.ply'
    print(f'Uploading {os.path.getsize(ply)/1e6:.1f} MB to Vercel Blob...')
    with open(ply, 'rb') as f:
        r = requests.put(
            f'https://blob.vercel-storage.com/{remote}',
            headers={
                'Authorization': f'Bearer {BLOB_TOKEN}',
                'x-content-type': 'model/vnd.ply',
                'x-cache-control-max-age': '31536000'
            },
            data=f, timeout=600
        )
    r.raise_for_status()
    url = r.json()['url']
    print('Uploaded:', url)
    print('SuperSplat:', f'https://superspl.at/editor?load={url}')
    print()
    print('To wire into the app: go to Supabase dashboard → tours table')
    print(f'  Set ply_url = {url}')
    print(f'  Set status  = complete')
    print(f'  Then open /tours/{TOUR_ID} in the app — the viewer should load.')
else:
    print('Set BLOB_TOKEN above to enable upload.')